# Задание
Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.

## 1. Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы.

In [ ]:
from pyspark.sql import Row
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
from pyspark.sql.window import Window

import os
import sys

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
xml_path = "/content/drive/MyDrive/posts_sample.xml"
csv_path = "/content/drive/MyDrive/programming-languages.csv"
result_path = "/content/programming_languages_report.parquet"
archive_name = "/content/programming_languages_report"

In [ ]:
spark = (
    SparkSession.builder
    .appName("programming_languages_statistics")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.2


Загрузка справочника

In [ ]:
csv_path = "/content/drive/MyDrive/programming-languages.csv"
languages_df = spark.read.csv(csv_path, header=True, inferSchema=True)
languages_df.show(5, truncate=False)
print("Число столбцов:", len(languages_df.columns))

+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
+----------+---------------------------------------------------------+
only showing top 5 rows
Число столбцов: 2


Подготовка списка языков

In [ ]:
first_col = languages_df.columns[0]

valid_languages = (
    languages_df
    .select(sf.lower(sf.trim(sf.col(first_col))).alias("lang"))
    .filter(sf.col("lang").isNotNull())
    .filter(sf.col("lang") != "")
    .distinct()
)

print("Количество уникальных языков:", valid_languages.count())
valid_languages.orderBy("lang").show(15, truncate=False)

Количество уникальных языков: 698
+----------+
|lang      |
+----------+
|@formula  |
|a# (axiom)|
|a# .net   |
|a+        |
|a++       |
|a-0 system|
|abap      |
|abc       |
|abc algol |
|abset     |
|absys     |
|acc       |
|accent    |
|ace dasl  |
|acl2      |
+----------+
only showing top 15 rows


Чтение XML-файла

In [ ]:
posts_raw = spark.read.text(xml_path).withColumnRenamed("value", "line")

posts_raw.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Отбор строк с датой и тегами

In [ ]:
posts_filtered = (
    posts_raw
    .filter(sf.col("line").contains('CreationDate="'))
    .filter(sf.col("line").contains('Tags="'))
)

print("Количество строк после первичной фильтрации:", posts_filtered.count())
posts_filtered.show(10, truncate=False)

Количество строк после первичной фильтрации: 18055
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Извлечение года и тегов

In [ ]:
posts_info = (
    posts_filtered
    .select(
        sf.regexp_extract("line", r'CreationDate="(\d{4})', 1).cast("int").alias("year"),
        sf.regexp_extract("line", r'Tags="([^"]+)"', 1).alias("tags_raw")
    )
    .filter((sf.col("year") >= 2010) & (sf.col("year") <= 2020))
)

print("Количество постов за период 2010–2020:", posts_info.count())
posts_info.show(10, truncate=False)

Количество постов за период 2010–2020: 17642
+----+--------------------------------------------------------------+
|year|tags_raw                                                      |
+----+--------------------------------------------------------------+
|2010|&lt;c++&gt;&lt;character-encoding&gt;                         |
|2010|&lt;sharepoint&gt;&lt;infopath&gt;                            |
|2010|&lt;iphone&gt;&lt;app-store&gt;&lt;in-app-purchase&gt;        |
|2010|&lt;symfony1&gt;&lt;schema&gt;&lt;doctrine&gt;&lt;fixtures&gt;|
|2010|&lt;java&gt;                                                  |
|2010|&lt;visual-studio-2010&gt;&lt;stylecop&gt;                    |
|2010|&lt;cakephp&gt;&lt;file-upload&gt;&lt;swfupload&gt;           |
|2010|&lt;git&gt;&lt;cygwin&gt;&lt;putty&gt;                        |
|2010|&lt;drupal&gt;&lt;drupal-6&gt;                                |
|2010|&lt;php&gt;&lt;wordpress&gt;&lt;memory&gt;                    |
+----+---------------------------------------

Очистка от лишних тегов и разбиение на отдельные значения

In [ ]:
tags_exploded = (
    posts_info
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", "&lt;", "<"))
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", "&gt;", ">"))
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", "&amp;", "&"))
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", "&quot;", '"'))
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", "&apos;", "'"))
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", "><", " "))
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", "<", ""))
    .withColumn("tags_raw", sf.regexp_replace("tags_raw", ">", ""))
    .withColumn("tag", sf.explode(sf.split(sf.trim(sf.col("tags_raw")), r"\s+")))
    .select(
        "year",
        sf.lower(sf.trim(sf.col("tag"))).alias("tag_name")
    )
    .filter(sf.col("tag_name") != "")
)

print("Количество тегов после нормализации:", tags_exploded.count())
tags_exploded.show(20, truncate=False)

Количество тегов после нормализации: 52118
+----+------------------+
|year|tag_name          |
+----+------------------+
|2010|c++               |
|2010|character-encoding|
|2010|sharepoint        |
|2010|infopath          |
|2010|iphone            |
|2010|app-store         |
|2010|in-app-purchase   |
|2010|symfony1          |
|2010|schema            |
|2010|doctrine          |
|2010|fixtures          |
|2010|java              |
|2010|visual-studio-2010|
|2010|stylecop          |
|2010|cakephp           |
|2010|file-upload       |
|2010|swfupload         |
|2010|git               |
|2010|cygwin            |
|2010|putty             |
+----+------------------+
only showing top 20 rows


Оставляю только те теги, которые являются языками программирования

In [ ]:
language_mentions = (
    tags_exploded.alias("t")
    .join(valid_languages.alias("l"), sf.col("t.tag_name") == sf.col("l.lang"), "inner")
    .groupBy(sf.col("t.year").alias("year"), sf.col("t.tag_name").alias("language"))
    .agg(sf.count("*").alias("mentions_count"))
)

language_mentions.orderBy("year", sf.desc("mentions_count")).show(30, truncate=False)

+----+------------+--------------+
|year|language    |mentions_count|
+----+------------+--------------+
|2010|java        |52            |
|2010|php         |46            |
|2010|javascript  |44            |
|2010|python      |26            |
|2010|objective-c |23            |
|2010|c           |20            |
|2010|ruby        |12            |
|2010|delphi      |8             |
|2010|applescript |3             |
|2010|r           |3             |
|2010|perl        |3             |
|2010|bash        |3             |
|2010|haskell     |2             |
|2010|f#          |2             |
|2010|sed         |2             |
|2010|matlab      |1             |
|2010|ocaml       |1             |
|2010|racket      |1             |
|2010|xpath       |1             |
|2010|go          |1             |
|2010|mouse       |1             |
|2010|powershell  |1             |
|2010|scheme      |1             |
|2010|actionscript|1             |
|2010|basic       |1             |
|2010|ksh         |1

Вычисление топ-10 по каждому году

In [ ]:
year_window = Window.partitionBy("year").orderBy(sf.desc("mentions_count"), sf.asc("language"))

top10_by_year = (
    language_mentions
    .withColumn("position", sf.row_number().over(year_window))
    .filter(sf.col("position") <= 10)
    .select("year", "position", "language", "mentions_count")
    .orderBy("year", "position")
)

for year in sorted(top10_by_year.select("year").distinct().rdd.flatMap(lambda x: x).collect()):
    print(f"\nТоп-10 языков для {year} года:")
    top10_by_year.filter(sf.col("year") == year).show(10, truncate=False)


Топ-10 языков для 2010 года:
+----+--------+-----------+--------------+
|year|position|language   |mentions_count|
+----+--------+-----------+--------------+
|2010|1       |java       |52            |
|2010|2       |php        |46            |
|2010|3       |javascript |44            |
|2010|4       |python     |26            |
|2010|5       |objective-c|23            |
|2010|6       |c          |20            |
|2010|7       |ruby       |12            |
|2010|8       |delphi     |8             |
|2010|9       |applescript|3             |
|2010|10      |bash       |3             |
+----+--------+-----------+--------------+


Топ-10 языков для 2011 года:
+----+--------+-----------+--------------+
|year|position|language   |mentions_count|
+----+--------+-----------+--------------+
|2011|1       |php        |102           |
|2011|2       |java       |93            |
|2011|3       |javascript |83            |
|2011|4       |python     |37            |
|2011|5       |objective-c|34       

Проверка на то, какие годы вошли в отчёт

In [ ]:
top10_by_year.select("year").distinct().orderBy("year").show(20, truncate=False)
print("Всего строк в итоговой таблице:", top10_by_year.count())

+----+
|year|
+----+
|2010|
|2011|
|2012|
|2013|
|2014|
|2015|
|2016|
|2017|
|2018|
|2019|
+----+

Всего строк в итоговой таблице: 100


#2. Сохранение результата в Parquet

In [ ]:
(
    top10_by_year
    .write
    .mode("overwrite")
    .parquet(result_path)
)

print("Parquet сохранён по пути:", result_path)

Parquet сохранён по пути: /content/programming_languages_report.parquet


Чтение из сохраненного файла

In [ ]:
check_df = spark.read.parquet(result_path)

check_df.orderBy("year", "position").show(120, truncate=False)
print("Количество строк в parquet:", check_df.count())

+----+--------+-----------+--------------+
|year|position|language   |mentions_count|
+----+--------+-----------+--------------+
|2010|1       |java       |52            |
|2010|2       |php        |46            |
|2010|3       |javascript |44            |
|2010|4       |python     |26            |
|2010|5       |objective-c|23            |
|2010|6       |c          |20            |
|2010|7       |ruby       |12            |
|2010|8       |delphi     |8             |
|2010|9       |applescript|3             |
|2010|10      |bash       |3             |
|2011|1       |php        |102           |
|2011|2       |java       |93            |
|2011|3       |javascript |83            |
|2011|4       |python     |37            |
|2011|5       |objective-c|34            |
|2011|6       |c          |24            |
|2011|7       |ruby       |20            |
|2011|8       |perl       |9             |
|2011|9       |delphi     |8             |
|2011|10      |bash       |7             |
|2012|1    

Архивирую

In [ ]:
import shutil

shutil.make_archive(
    archive_name,
    "zip",
    result_path
)

print("Архив создан:", archive_name + ".zip")

Архив создан: /content/programming_languages_report.zip


Скачивание архива в Colab

In [ ]:
from google.colab import files
files.download("programming_languages_report.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
spark.stop()